In [1]:
import numpy as np
import soundfile as sf
from scipy.signal import fftconvolve
import os

In [2]:
import os
print("CWD:", os.getcwd())
print("Files here:", os.listdir(".")[:20])

CWD: /Users/liann77/Desktop/NYUStudy/NYU26Spring/Colloquy/colluquyTeamWork/Surround-Stereo-Binaural-Custom-BRIR-based-Converter-/HRTF_Processor
Files here: ['.DS_Store', 'TEST_AUDIO_44k.wav', 'Code.ipynb', 'Source', 'BRIR', '48K_24bit', 'extract_brir.ipynb']


In [3]:
import soundfile as sf
from scipy.signal import resample_poly
import numpy as np
import math

# 读取 test audio
audio, sr = sf.read("Source/TEST_AUDIO.wav")

print("Original sample rate:", sr)

target_sr = 44100

if sr != target_sr:

    gcd = math.gcd(sr, target_sr)
    up = target_sr // gcd
    down = sr // gcd

    if audio.ndim == 1:
        audio_resampled = resample_poly(audio, up, down)
    else:
        audio_resampled = np.stack(
            [resample_poly(audio[:,i], up, down) for i in range(audio.shape[1])],
            axis=1
        )

    sf.write("Source/TEST_AUDIO_44k.wav", audio_resampled, target_sr)

    print("Downsampled to 44.1k and saved as TEST_AUDIO_44k.wav")

else:
    print("Audio already at 44.1k")

Original sample rate: 48000
Downsampled to 44.1k and saved as TEST_AUDIO_44k.wav


In [4]:
sweep_file = "Source/TEST_AUDIO_44k.wav"

sweep, fs = sf.read(sweep_file)

print("Sweep length:", len(sweep))
print("Sample rate:", fs)

Sweep length: 441002
Sample rate: 44100


In [5]:
inverse_sweep = sweep[::-1]

inverse_sweep = inverse_sweep / np.max(np.abs(inverse_sweep))

print("Inverse sweep created")

Inverse sweep created


In [6]:
def extract_brir(record_file, inverse_sweep, fs, output_file):

    rec, fs_rec = sf.read(record_file)

    if fs_rec != fs:
        raise ValueError("Sample rate mismatch")

    print("Processing:", record_file)

    # dummy head recording (stereo)
    left = fftconvolve(rec[:,0], inverse_sweep, mode='full')
    right = fftconvolve(rec[:,1], inverse_sweep, mode='full')

    brir = np.stack([left, right], axis=1)

    # normalize
    brir = brir / np.max(np.abs(brir))

    sf.write(output_file, brir, fs)

    print("Saved:", output_file)

In [12]:
recordings = {
    "CenterBRIR2.wav": "BRIR_C.wav",
    "LeftBRIR2.wav": "BRIR_L.wav",
    "RightBRIR2.wav": "BRIR_R.wav",
    "LeftRearBRIR2.wav": "BRIR_LS.wav",
    "RightRearBRIR2.wav": "BRIR_RS.wav",
    "SubBRIR2.wav": "BRIR_SUB.wav",
}

recording_folder = "Source/BRIRRecordings"
output_folder = "BRIR"

os.makedirs(output_folder, exist_ok=True)

for rec, out in recordings.items():
    rec_path = os.path.join(recording_folder, rec)
    out_path = os.path.join(output_folder, out)

    print("Processing:", rec_path)

    extract_brir(rec_path, inverse_sweep, fs, out_path)

Processing: Source/BRIRRecordings/CenterBRIR2.wav
Processing: Source/BRIRRecordings/CenterBRIR2.wav
Saved: BRIR/BRIR_C.wav
Processing: Source/BRIRRecordings/LeftBRIR2.wav
Processing: Source/BRIRRecordings/LeftBRIR2.wav
Saved: BRIR/BRIR_L.wav
Processing: Source/BRIRRecordings/RightBRIR2.wav
Processing: Source/BRIRRecordings/RightBRIR2.wav
Saved: BRIR/BRIR_R.wav
Processing: Source/BRIRRecordings/LeftRearBRIR2.wav
Processing: Source/BRIRRecordings/LeftRearBRIR2.wav
Saved: BRIR/BRIR_LS.wav
Processing: Source/BRIRRecordings/RightRearBRIR2.wav
Processing: Source/BRIRRecordings/RightRearBRIR2.wav
Saved: BRIR/BRIR_RS.wav
Processing: Source/BRIRRecordings/SubBRIR2.wav
Processing: Source/BRIRRecordings/SubBRIR2.wav
Saved: BRIR/BRIR_SUB.wav
